# M-Optimus: Spatial Transcriptomics from Histology

**Predict spatial gene expression directly from H&E slides — optionally refined
with bulk RNA-seq.**

---

## What this notebook does

[M-Optimus](https://docs.bioptimus.com/documentation/models/m-optimus) is a
multimodal, multi-scale foundation model that learns across three biological
layers: H&E pathology, bulk RNA-seq, and spatial transcriptomics. Its headline
capability is predicting **spatial gene expression** 
directly from a routine H&E tile, recovering an expensive molecular readout
from a low-cost slide.

M-Optimus also produces embeddings of the same dimension (1536) as [H-Optimus-1](https://docs.bioptimus.com/documentation/models/h-optimus); like with H-Optimus-1, these embeddings can also be used for morphological and molecular prediction.

| Capability | Output |
|---|---|
| **Tile embeddings** | 1536-dimensional feature vector (same as H-Optimus-1) |
| **Spatial gene expression** | Predicted expression for ~6,000 genes per tile |
| **Multimodal refinement** | Optional bulk RNA input improves predictions ~4% |

### Predicted gene panel

M-Optimus predicts expression for a curated set of **~6,000 genes** per
tile. This is not a genome-wide prediction — the gene panel was selected
for biological relevance and prediction quality across tissue types.
Output gene names are Ensembl IDs (e.g. `ENSG00000141510`). The full list
is available at runtime via `model.output_gene_names` (low-level API) or
in the `gene_names` array of the Zarr output.

This notebook demonstrates the **complete spatial transcriptomics pipeline**:

1. Download a TCGA-LUAD slide and matched bulk RNA-seq data
2. Run **image-only** predictions (no bulk RNA)
3. **Late-bind** bulk RNA and re-predict (multimodal)
4. Compare predictions before and after RNA context
5. Visualize spatial gene expression heatmaps for biologically relevant genes

---

### Documentation links

| Resource | URL |
|---|---|
| M-Optimus model page | [docs.bioptimus.com/documentation/models/m-optimus](https://docs.bioptimus.com/documentation/models/m-optimus) |
| Spatial transcriptomics guide | [docs.bioptimus.com/guides/workflows/spatial-transcriptomics](https://docs.bioptimus.com/guides/workflows/spatial-transcriptomics) |
| SDK overview | [docs.bioptimus.com/guides/get-started/sdk](https://docs.bioptimus.com/guides/get-started/sdk) |
| Inference facade | [docs.bioptimus.com/guides/get-started/inference-facade](https://docs.bioptimus.com/guides/get-started/inference-facade) |
| Cohorts guide | [docs.bioptimus.com/guides/workflows/cohort](https://docs.bioptimus.com/guides/workflows/cohort) |
| Visualizing results | [docs.bioptimus.com/guides/get-started/visualizing-results](https://docs.bioptimus.com/guides/get-started/visualizing-results) |
| Choosing a model | [docs.bioptimus.com/documentation/models/choosing-a-model](https://docs.bioptimus.com/documentation/models/choosing-a-model) |

## Prerequisites

| Requirement | Details |
|---|---|
| **Python** | 3.12+ (pre-installed in this environment) |
| **SDK** | `bioptimus-sdk` (installed in the cell below) |
| **Inference backend** | One of: a running Bioptimus server (`remote`), a SageMaker endpoint (`aws`), or local GPU with `.pt2` checkpoints (`local`) |
| **Input data** | Downloaded automatically: one `.svs` slide + one `.tsv` RNA-seq file for TCGA-LUAD |
| **Disk space** | ~2 GB for the slide, ~1 MB for RNA, ~500 MB for outputs |
| **Python packages** | `numpy`, `zarr`, `requests`, `pandas`, `scikit-learn`, `matplotlib` |

> **Local backend additional requirements:** A CUDA GPU compatible with the
> exported `.pt2` models (default: `sm_86` — A10G, RTX 3090),
>
> `pip install bioptimus-sdk[torch]` (installs `torch` and `torchvision`),
> and required model artifacts such as input and output gene lists.

## 0. Environment Setup

Quiets noisy startup warnings, verifies the Python version, and installs the
Bioptimus SDK with visualization dependencies.

In [ ]:
# --- Logging / warning configuration ----------------------------------------
# Quiet noisy PyTorch and Python
# warnings unless LOG_LEVEL=DEBUG. Runs before torch / the SDK is imported so
# that TORCH_LOGS and TORCH_CPP_LOG_LEVEL take effect. Export any variable (or
# set LOG_LEVEL=DEBUG) before launching Jupyter to override these defaults.
import os
import warnings

LOG_LEVEL = os.environ.setdefault("LOG_LEVEL", "INFO")

if LOG_LEVEL == "DEBUG":
    os.environ.setdefault("PYTHONWARNINGS", "default")
    os.environ.setdefault("TORCH_LOGS", "+inductor")
    os.environ.setdefault("TORCH_CPP_LOG_LEVEL", "WARNING")
else:
    os.environ.setdefault("PYTHONWARNINGS", "ignore::FutureWarning,ignore::UserWarning")
    os.environ.setdefault("TORCH_LOGS", "-all")
    os.environ.setdefault("TORCH_CPP_LOG_LEVEL", "ERROR")
    # PYTHONWARNINGS only affects subprocesses and warnings not yet triggered,
    # so also filter the running kernel process.
    warnings.filterwarnings("ignore", category=FutureWarning)
    warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
import importlib.metadata
import sys

assert sys.version_info >= (3, 12), f"Python 3.12+ is required, but found {sys.version}."
print(f"Python:     {sys.version}")
print(f"Executable: {sys.executable}")

# If the SDK is not already installed, uncomment the line below and run this cell.
# The [torch] extra is REQUIRED for the local (in-process GPU) backend that this
# notebook uses by default; it installs torch and torchvision.
# %pip install -q "bioptimus-sdk[torch]" pandas scikit-learn matplotlib

try:
    sdk_version = importlib.metadata.version("bioptimus-sdk")
    print(f"Bioptimus:  {sdk_version}")
except importlib.metadata.PackageNotFoundError:
    print("Bioptimus:  Not installed — uncomment the %pip line above and re-run this cell.")


### Imports

| Import | Purpose | Docs |
|---|---|---|
| `Inference` | High-level pipeline facade — tissue masking, embedding, prediction, workspace | [Inference facade](https://docs.bioptimus.com/guides/get-started/inference-facade) |
| `Cohort` | Multi-slide experiment manifest — tracks slides, RNA data, masks, and outputs | [Cohorts](https://docs.bioptimus.com/guides/workflows/cohort) |
| `Models` | Enum of available model names (`H1`, `M_OPTIMUS`, `TISSUE_SEG`) | [Choosing a model](https://docs.bioptimus.com/documentation/models/choosing-a-model) |
| `Backend` | Enum of inference backends (`REMOTE`, `AWS`, `LOCAL`) | [SDK overview](https://docs.bioptimus.com/guides/get-started/sdk) |
| `bioptimus.utils` | Shared helpers for download, visualization, and output loading | — |


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from bioptimus import utils
from bioptimus.data.cohort import Cohort
from bioptimus.inference import Inference
from bioptimus.models import Backend, Models

---

## 1. Demo Data Setup

We download two files for TCGA-LUAD sample **TCGA-75-7027**
([GDC case page](https://portal.gdc.cancer.gov/cases/488646a1-0bdb-45f8-8c30-bef006924511)):

| File | Format | Purpose |
|---|---|---|
| Diagnostic slide | `.svs` (~2 GB) | H&E whole-slide image |
| Gene expression | `.tsv` (~1 MB) | Bulk RNA-seq quantification (TPM) |

The bulk RNA file is **not linked yet** — we add it later (Section 8) to
demonstrate **late-binding**: the ability to first run image-only predictions,
then attach transcriptomic context and re-predict without retraining.

> **Note:** Downloads are idempotent — re-running skips existing files.

In [ ]:
# --- Demo data configuration ------------------------------------------------
DEMO_DATA_DIR = Path.cwd() / "demo_external_data"
DEMO_WSI_DIR = DEMO_DATA_DIR / "wsi"
DEMO_OMICS_DIR = DEMO_DATA_DIR / "omics"

# TCGA-LUAD sample TCGA-75-7027 (diagnostic slide + RNA-seq).
SAMPLE_NAME = "TCGA-75-7027"
SVS_URL = "https://api.gdc.cancer.gov/data/3614fe76-11a3-4b95-a0d6-426ec32fe90c"
TSV_URL = "https://api.gdc.cancer.gov/data/b7b898c7-f63f-43d3-864a-406e842db2fe"

DEMO_WSI_DIR.mkdir(parents=True, exist_ok=True)
DEMO_OMICS_DIR.mkdir(parents=True, exist_ok=True)

svs_output_path = DEMO_WSI_DIR / f"{SAMPLE_NAME}.svs"
tsv_output_path = DEMO_OMICS_DIR / f"{SAMPLE_NAME}.tsv"

print("Downloading demo data...")
utils.download_if_needed(SVS_URL, svs_output_path)
utils.download_if_needed(TSV_URL, tsv_output_path)

print(f"\nWSI directory:   {DEMO_WSI_DIR}")
print(f"Omics directory: {DEMO_OMICS_DIR}")

### Sanity check — slide and RNA inspection

Always inspect your inputs before running a pipeline:
- **Slide:** Verify the thumbnail matches the
  [GDC case page](https://portal.gdc.cancer.gov/cases/488646a1-0bdb-45f8-8c30-bef006924511),
  and check the MPP / dimensions.
- **RNA:** Confirm the gene count, column layout, and expression distribution.

The SDK's [`WSI`](https://docs.bioptimus.com/guides/reference/wsi-reader) reader
auto-detects metadata from the slide header.
`utils.display_slide_info()` accepts an optional `thumbnail_size=(w, h)` to
control the thumbnail resolution (default `(512, 512)`).

In [ ]:
# Slide metadata + thumbnail.
print("=" * 50)
print("SLIDE")
print("=" * 50)
wsi_thumbnail = utils.display_slide_info(svs_output_path)

# RNA-seq summary.
print()
print("=" * 50)
print("BULK RNA-SEQ")
print("=" * 50)
rna_df = utils.display_rna_summary(tsv_output_path)

# Show thumbnail.
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(wsi_thumbnail)
ax.set_title(f"Slide thumbnail — {SAMPLE_NAME}")
ax.axis("off")
plt.tight_layout()
plt.show()

---

## 2. Configuration

Configure the inference backend and experiment parameters.

The SDK supports three backends. **This notebook is configured to use the
`local` backend by default** (in-process GPU, no server needed) — see the
code cell below to switch.

| Backend | When to use | Key parameters |
|---|---|---|
| `"remote"` | You have a running Bioptimus FastAPI server (Docker or bare-metal) | `api_url` |
| `"aws"` | You have a deployed SageMaker endpoint | `endpoint_name`, `region_name` |
| `"local"` | You have a CUDA GPU, the `.pt2` checkpoints, and the assets file (no server) | `checkpoints`, `device`, `assets_root` |

> **Not sure which to pick?** If you were given a server URL, use `remote`.
> If you deployed on AWS, use `aws`. If you just have a GPU machine plus the
> model files, use `local` (the default here).

The [`Inference`](https://docs.bioptimus.com/guides/get-started/inference-facade)
facade writes outputs into a structured workspace:

```
<output_path>/<experiment>/run_<run>/<variant>/
    config.yaml
    tissue/<slide>.png
    m-optimus/embeddings/<slide>.zarr
    m-optimus/predictions/<slide>.zarr
```

> **Tip:** Use the `variant` parameter to compare different setups
> (e.g. `variant="fp32"` vs. `variant="fp16"`) without overwriting results.

> ⚠️ **Before you run the next cell — fill in your settings.**
> The cell below uses the `local` backend, which needs model files on disk.
> You **must** set these three paths:
> - `M_OPTIMUS_CHECKPOINT` — path to the M-Optimus `.pt2` checkpoint
> - `TISSUE_SEG_CHECKPOINT` — path to the tissue-segmentation `.pt2` checkpoint
> - `ASSETS_ROOT` — path to the M-Optimus **assets CSV**, which lists the input
>   and output genes the model expects (M-Optimus needs this to align the gene
>   columns; H-Optimus-1 does not). It ships alongside the checkpoints.
>
> The remaining settings (`OUTPUT_DIR`, `EXPERIMENT`, `VARIANT`, `RUN`) come with
> working defaults — keep them for a first run, or change them to organize your
> results. **Where do these files come from?** They are provided by Bioptimus. If you don't
> have them, use the `remote` or `aws` backend instead.


In [ ]:
# --- Experiment configuration -------------------------------------------------
# Choose ONE backend. The `local` backend (Option 3) is ACTIVE BY DEFAULT below.
# To use `remote` or `aws` instead, uncomment that option here AND update the
# matching block in the `Inference(...)` call in Section 4.

# Option 1: Remote server (HTTP).
# API_URL = "http://0.0.0.0:8080"
# utils.check_server(API_URL)

# Option 2: AWS SageMaker — uncomment and set your endpoint:
# ENDPOINT_NAME = "your-m-optimus-endpoint"
# REGION_NAME = "us-east-1"

# Option 3: Local in-process GPU inference (no server required) — ACTIVE DEFAULT.
# Requires .pt2 checkpoints, the assets CSV, and: pip install "bioptimus-sdk[torch]".
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "The local backend requires a CUDA GPU, but none was found. "
        "Either run on a GPU machine, or switch to the remote/aws backend "
        "(comment out this Option 3 block and uncomment Option 1 or 2)."
    )

num_gpus = torch.cuda.device_count()
print(f"Available GPUs: {num_gpus}")
for i in range(num_gpus):
    print(f"  cuda:{i} — {torch.cuda.get_device_name(i)}")

# Select which GPU to use (0-indexed). Each process uses a single GPU.
# For multi-GPU parallelism, run separate processes with different GPU_ID values.
GPU_ID = 0
DEVICE = f"cuda:{GPU_ID}"
print(f"\nUsing: {DEVICE} ({torch.cuda.get_device_name(GPU_ID)})")

# --- REQUIRED: set these to real files on disk ------------------------------
# Provided by Bioptimus.
# ASSETS_ROOT points to the M-Optimus assets CSV (input/output gene lists).
# The dict keys ("m-optimus", "tissue-seg") are fixed identifiers — do not rename.
M_OPTIMUS_CHECKPOINT = "<PATH TO M-OPTIMUS CHECKPOINT>"
TISSUE_SEG_CHECKPOINT = "<PATH TO TISSUE SEGMENTATION CHECKPOINT>"
ASSETS_ROOT = "<PATH TO ASSETS CSV>"
CHECKPOINTS = {
    "m-optimus": str(M_OPTIMUS_CHECKPOINT),
    "tissue-seg": str(TISSUE_SEG_CHECKPOINT),
}

# --- Experiment settings (sensible defaults — change them if you like) ------
WSI_DIR = DEMO_WSI_DIR
BULK_RNA_DIR = DEMO_OMICS_DIR
OUTPUT_DIR = Path("<PATH TO OUTPUT DIRECTORY>") # where results are written
EXPERIMENT = "<EXPERIMENT NAME>" # a name for this experiment
VARIANT = "<VARIANT NAME>" # label for this configuration 
RUN = "<RUN NUMBER>" # a unique identifier for this run to keep results separate
MASK_THRESHOLD = 0.5


---

## 3. Build a Cohort (Image Only)

We build the cohort from WSIs **without** linking bulk RNA yet. This is
intentional — M-Optimus supports **late-binding**: you can run image-only
predictions first, then attach RNA data and re-predict to see the improvement.

### Cohort construction rules

[`Cohort.from_directories()`](https://docs.bioptimus.com/guides/workflows/cohort)
scans a directory for WSI files and creates a `WSIRecord` per slide.
Matching between WSIs, bulk RNA files, and masks is done by **exact
filename stem** (case-sensitive):

```
wsi_dir/
    TCGA-75-7027.svs      ← stem = "TCGA-75-7027"
bulk_rna_dir/
    TCGA-75-7027.tsv      ← matches WSI above
mask_dir/
    TCGA-75-7027.png      ← matches WSI above
```

Key rules:
- Supported WSI extensions: `.svs`, `.tiff`, `.tif`, `.ndpi`, `.mrxs`,
  `.vms`, `.vsi`, `.scn`, `.jp2` (via `WSI.supported_extensions()`).
- If two slides share the same stem but different extensions (e.g.
  `slide.svs` and `slide.tiff`), both are registered — avoid this.
- Patient ID defaults to the filename stem. Pass `patient_id_fn` to
  extract a custom ID (e.g. strip the tissue type suffix).
- When multiple WSIs map to the same patient ID, they are assigned
  sequential timepoints (`t0`, `t1`, …) in alphabetical order.
- Unmatched WSIs (no corresponding RNA file) proceed as image-only.
  Unmatched RNA files are logged as warnings.

Each record tracks:
- The path to the WSI file
- The patient ID (extracted from the filename)
- Optional bulk RNA, mask, and output paths
- Processing status per model and stage

Other options for building or managing cohorts:
- `Cohort.from_directories(wsi_dir, bulk_rna_dir=..., mask_dir=..., patient_id_fn=...)`
  — auto-pair WSIs with RNA and masks.
- `Cohort.from_csv(csv_path)` — build from a CSV manifest.
- `cohort.save("manifest.yaml")` / `Cohort.load("manifest.yaml")` — persist
  and reload a cohort.

> At this point, `available_modalities` should show `["image"]` only.

In [ ]:
# Build cohort from WSIs only — no bulk RNA linked yet.
cohort = Cohort.from_directories(wsi_dir=WSI_DIR)

print(cohort.summary())
print(f"\nWSI IDs:    {cohort.wsi_ids}")
print(f"Patients:   {cohort.patient_ids}")
print(f"Modalities: {cohort[0].available_modalities}  (image only — RNA linked later)")

---

## 4. Create the Inference Pipeline

The [`Inference`](https://docs.bioptimus.com/guides/get-started/inference-facade)
facade manages the full pipeline for M-Optimus. Compared to H-Optimus-1,
M-Optimus supports two modes:

| Mode | Method | Output |
|---|---|---|
| `"embed"` | `infer.run(mode="embed")` | 1536-dimensional tile embeddings |
| `"predict"` | `infer.run(mode="predict")` | Spatial gene expression (`num_tiles x num_genes` dimensional output, with `num_genes` ~ 6k) |

Bulk RNA usage is **automatic**: if a cohort record has `bulk_rna_path` set,
the model receives it; otherwise it runs image-only. Each output is tagged
with the modalities used (e.g. `["image"]` or `["image", "bulk_rna"]`).

Additional optional parameters:
- `output_format="zarr"` — also supports `"hdf5"` and `"npz"`.
- `max_concurrency=256` — max concurrent tile requests to the server.
- `wsi_backend="cucim"` — force a specific WSI reader (`"cucim"`,
  `"openslide"`, `"tifffile"`).
- `description="..."` — free-text description saved in the workspace config.

> **Resume a previous run:** `Inference.from_workspace("path/to/workspace")`
> reloads the config and cohort manifest from a saved workspace.

> **Backend selection:** The code cell below uses the `local` backend.
> See the commented sections to switch to `remote` (HTTP server) or
> `aws` (SageMaker). For `local`, pass `checkpoints`, `device`, and
> `assets_root`. See the
> [local backend tutorial](../docs/sdk-tutorial-local.md) for full details.


In [ ]:
# --- Create the Inference pipeline ---------------------------------------------
# The `local` backend block is ACTIVE BY DEFAULT below. To use remote or aws,
# comment out the "Option 3: Local" lines and uncomment the matching block
# (and set its variables in Section 2). Only ONE backend may be active at a time.

infer = Inference(
    model_name=Models.M_OPTIMUS,
    # --- Option 1: Remote server (HTTP) ---
    # backend=Backend.REMOTE,
    # api_url=API_URL,
    # timeout=120.0,
    # --- Option 2: AWS SageMaker ---
    # backend=Backend.AWS,
    # endpoint_name=ENDPOINT_NAME,
    # region_name=REGION_NAME,
    # timeout=120.0,
    # --- Option 3: Local in-process GPU (no server) — ACTIVE DEFAULT ---
    backend=Backend.LOCAL,
    checkpoints=CHECKPOINTS,
    device=DEVICE,
    assets_root=ASSETS_ROOT,
    # --- Common parameters (all backends) ---
    cohort=cohort,
    tissue=True,
    mask_threshold=MASK_THRESHOLD,
    output_path=OUTPUT_DIR,
    experiment=EXPERIMENT,
    variant=VARIANT,
    run=RUN,
    workers=1,
)

print(f"Model:     {infer.model_name}")
print(f"Workspace: {infer.workspace}")
print(f"WSIs:      {len(cohort)}")


---

## 5. Tissue Mask Generation

Tissue masking identifies tissue-bearing tiles and discards background. The
bundled [tissue segmentation model](https://docs.bioptimus.com/documentation/models/tissue-segmentation)
tiles the slide coarsely (512×512 at 8 µm/px) and classifies each tile.

Masks are **shared across models** — if you already generated masks with
H-Optimus-1 (see `h_optimus_1_tutorial.ipynb`), they will be reused
automatically.

> **All backends:** Tissue masking works identically on `remote`, `aws`, and
> `local` backends. For `local`, the tissue-seg `.pt2` checkpoint is resolved
> from the `checkpoints` dict passed to `Inference`.

> **Caching:** `infer.tissue()` skips slides that already have masks on disk.
> Pass `force=True` to recompute, or a single `wsi_path` to process just
> one slide.

In [ ]:
# Generate tissue masks (resumes from where it left off).
infer.tissue()

print(cohort.summary())

In [ ]:
# Visualize the tissue mask alongside the slide thumbnail.
first_record = cohort[0]

utils.plot_slide_and_mask(
    wsi_path=first_record.wsi_path,
    mask_path=first_record.mask_path,
    threshold=MASK_THRESHOLD,
)


---

## 6. Extract M-Optimus Embeddings

[M-Optimus](https://docs.bioptimus.com/documentation/models/m-optimus) produces embeddings of the same dimensionality (1536) as [H-Optimus-1](https://docs.bioptimus.com/documentation/models/h-optimus).
Running `mode="embed"` extracts features without gene-expression predictions.

This is useful when you want both morphological features **and** spatial
transcriptomics from the same model — embeddings are always available
regardless of whether you run predictions later.

Outputs are written to `<workspace>/m-optimus/embeddings/<slide>.zarr`.
See [Visualizing results](https://docs.bioptimus.com/guides/get-started/visualizing-results)
for the output schema, and [Tile embeddings & PCA](https://docs.bioptimus.com/guides/workflows/embeddings-pca)
for downstream analysis.

> Pass `force=True` to `infer.run()` to reprocess slides that already have
> outputs on disk.

> **Troubleshooting:** After a run, `infer.failures` returns a list of any
> WSIs that encountered errors. Use `infer.status()` to print an overview
> of cohort processing progress.

In [ ]:
# Extract M-Optimus embeddings.
#
# NOTE on re-running this whole notebook: Section 8 links bulk RNA to the cohort.
# On a second full run the cohort may still have bulk RNA attached, so embeddings
# would then use image + bulk RNA. To force a clean image-only run, unlink bulk
# RNA first by uncommenting the two lines below:
# for record in cohort:
#     record.bulk_rna_path = None

infer.run(mode="embed")

print(cohort.summary())

# Verify output dimensions — explicitly load the image-only embeddings so that
# re-running this cell after linking bulk RNA still fetches the correct output.
first_record = cohort[0]
m_output = first_record.outputs.get(infer.model_name)
embed_path = m_output.get_stage_path("embed", ["image"])
embeddings = utils.load_zarr_output(embed_path)["outputs"]
num_tiles = embeddings.shape[0]

# M-Optimus embeddings are always 1536-dimensional (same as H-Optimus-1).
assert embeddings.shape == (num_tiles, 1536), (
    f"Expected (n_tiles, 1536) embeddings, but got {embeddings.shape}."
)
print(f"\n✓ Embeddings: {num_tiles} tiles × 1536 dims")


---

## 7. Image-Only Predictions (Baseline)

Now we run the [prediction head](https://docs.bioptimus.com/documentation/models/m-optimus#modes)
**without bulk RNA** to establish a baseline.
For each tissue tile, M-Optimus predicts expression of ~6,000 genes from
the H&E image alone.

This is the core capability: **spatial transcriptomics without a spatial
assay**. The image-only predictions already capture tissue-type-specific
expression patterns, immune infiltration, and stromal signatures.

We save these predictions in memory for comparison after adding bulk RNA
in the next section.

`utils.load_zarr_output()` returns a dictionary of outputs with all available arrays and
metadata:

| Key | Shape | Description |
|---|---|---|
| `"outputs"` | `(n_tiles, n_genes)` | Predicted gene expression per tile |
| `"coords"` | `(n_tiles, 2)` | `(x, y)` pixel coordinates of each tile |
| `"metadata"` | dict | Slide attributes (`slide_name`, `tile_size`, `mpp`, etc.) |
| `"gene_names"` | list | Output gene list (Ensembl IDs) — `mode="predict"` only |
| `"tissue_ratios"` | `(n_tiles,)` | Tissue fraction per tile (if stored) |
| `"thumbnail"` | `(H, W, 3)` | Slide thumbnail for overlay visualization (if stored) |
| `"tissue_mask"` | `(H, W)` | Tissue mask used during extraction (if stored) |


> See the [Spatial transcriptomics guide](https://docs.bioptimus.com/guides/workflows/spatial-transcriptomics)
> for the full M-Optimus prediction workflow, and
> [Visualizing results](https://docs.bioptimus.com/guides/get-started/visualizing-results)
> for the output file schema.

In [ ]:
# Run predictions from images only (no bulk RNA).
infer.run(mode="predict")

# Load image-only predictions for later comparison.
m_output = first_record.outputs.get(infer.model_name)
img_only_path = m_output.get_stage_path("predict", ["image"])
print(f"Image-only prediction path: {img_only_path}")

data_img_only = utils.load_zarr_output(img_only_path)
preds_image_only = data_img_only["outputs"]
coords_image_only = data_img_only["coords"]
gene_names = data_img_only.get("gene_names", [])

print(f"Predictions shape: {preds_image_only.shape}  (tiles × genes)")
print(f"Output genes:      {len(gene_names):,}")
if gene_names:
    print(f"First 5 genes:     {gene_names[:5]}")

# M-Optimus predicts 6,002 genes per tile.
# We check the tile count and that the prediction width matches the gene list,
# rather than hard-coding the exact gene count, which may change across versions.
assert preds_image_only.shape[0] == num_tiles, (
    f"Expected {num_tiles} tiles, but got {preds_image_only.shape[0]}."
)
if gene_names:
    assert preds_image_only.shape[1] == len(gene_names), (
        f"Prediction width ({preds_image_only.shape[1]}) does not match "
        f"the number of gene names ({len(gene_names)})."
    )
print(f"\n✓ Predictions: {preds_image_only.shape[0]} tiles × {preds_image_only.shape[1]} genes")

print()
infer.report()


---

## 8. Late-Binding — Link Bulk RNA and Re-predict

M-Optimus can use bulk RNA-seq data as additional input when embedding or making predictions.
**Late-binding** means you can link the bulk expression after building
the cohort — no need to reconfigure or re-run the full pipeline.

### Expected bulk RNA input format

The input file must be a **TSV or CSV** with at least two columns:

| Column | Content |
|---|---|
| Gene ID column | Ensembl gene IDs (e.g. `ENSG00000141510`) |
| Value column | **TPM-normalized** expression values (not raw counts) |

The model expects TPM (transcripts per million) as input — raw counts or
FPKM will produce incorrect results. For TCGA/GDC files, use the
`tpm_unstranded` column.

Example file structure (TCGA GDC format):
```
gene_id             gene_name   tpm_unstranded  ...
ENSG00000000003.15  TSPAN6      11.4962         ...
ENSG00000000005.6   TNMD        0.0216          ...
```

### How `link_bulk_rna()` works

[`Cohort.link_bulk_rna()`](https://docs.bioptimus.com/guides/workflows/spatial-transcriptomics#3-add-bulk-rna-multimodal-prediction)
scans a directory for `.tsv`/`.csv` files and matches them to WSI records
by filename stem (e.g. `TCGA-75-7027.tsv` → `TCGA-75-7027.svs`).

For TCGA/GDC gene quantification files, you typically need:

| Parameter | Value | Why |
|---|---|---|
| `gene_column` | `"gene_id"` | GDC files use `gene_id` for Ensembl IDs |
| `value_column` | `"tpm_unstranded"` | TPM-normalized expression values |
| `strip_version` | `True` | Converts `ENSG00000000003.15` → `ENSG00000000003` |

Additional `link_bulk_rna()` options:
- `extensions` — file extensions to match (default `{".csv", ".tsv"}`).
- `separator` — column delimiter; auto-detected from extension by default.

The SDK automatically aligns and reorders genes to the model's expected
input set (a `log1p` transform is applied internally).

> **Output separation:** Image-only predictions are written to
> `predictions/<slide>.zarr`, while multimodal predictions go to
> `predictions_bulk_rna/<slide>.zarr`. Both are preserved — no need for
> `force=True`.

> **Impact:** Adding bulk RNA typically improves predictions by ~4% over
> image-only, with the largest gains in lowly-expressed genes.

In [ ]:
# Link bulk RNA files to the cohort by matching filename stems.
linked = cohort.link_bulk_rna(
    BULK_RNA_DIR,
    gene_column="gene_id",
    value_column="tpm_unstranded",
    strip_version=True,
)

print(f"Linked {linked} record(s) with bulk RNA.")
print(f"Modalities: {cohort[0].available_modalities}")

In [ ]:
# Re-run predictions with bulk RNA context.
# The SDK detects the new modality combination ("image" + "bulk_rna") and writes
# to a separate directory (predictions_bulk_rna/), preserving image-only outputs.
infer.run(mode="predict")

# Verify that the modalities tag has been updated.
m_output = first_record.outputs.get(infer.model_name)
print(f"Modalities used: {m_output.modalities}")
print()
infer.report()

---

## 9. Embedding PCA — Morphological Structure

Before comparing predictions, let's verify that
[M-Optimus](https://docs.bioptimus.com/documentation/models/m-optimus)
embeddings capture meaningful morphological variation (same analysis as in the
[H-Optimus-1 tutorial](./h_optimus_1_tutorial.ipynb)).

`utils.plot_pca_scatter()` options:
- `n_components` — number of PCA components (default `2`).
- `cmap` — colormap (`"viridis"`, `"plasma"`, `"coolwarm"`, `"tab10"`, etc.).
- `point_size` — scatter point size (default `10`).
- `figsize` — figure dimensions (default `(8, 6)`).

`utils.plot_spatial_pca()` options:
- `components` — which PCs to plot, 0-indexed (default: first 3).
- `figsize` — figure dimensions (default `(18, 5)`).

See [Tile embeddings & PCA](https://docs.bioptimus.com/guides/workflows/embeddings-pca)
for guidance on interpreting principal components, and
[Visualizing results](https://docs.bioptimus.com/guides/get-started/visualizing-results)
for the full output schema.

In [ ]:
# Load M-Optimus embeddings — explicitly request the image-only embeddings
# so this cell works correctly regardless of whether bulk RNA is linked.
embed_path = m_output.get_stage_path("embed", ["image"])
embed_data = utils.load_zarr_output(embed_path)
embeddings = embed_data["outputs"]
coords_embed = embed_data["coords"]

print(f"Embeddings: {embeddings.shape}  (tiles × dim)")

assert embeddings.shape == (num_tiles, 1536)

# PCA — compute 3 components; scatter shows PC1 vs PC2, spatial maps show all 3.
scores, pca = utils.plot_pca_scatter(
    embeddings,
    n_components=3,
    title=f"M-Optimus Embeddings PCA — {first_record.wsi_id}",
    cmap="plasma",
)

# Spatial PCA heatmaps (3 components — each captures a distinct tissue compartment).
utils.plot_spatial_pca(
    scores,
    coords_embed,
    components=[0, 1, 2],
    title_prefix=f"{first_record.wsi_id} — ",
)

### Visual Proof — Nearest-Neighbor Tiles

Do tiles with similar M-Optimus embeddings actually look alike? For each query
tile (sampled from different PCA regions), we show its closest neighbors in
cosine distance. Since
[M-Optimus](https://docs.bioptimus.com/documentation/models/m-optimus) is
trained on both histology and molecular data, its embeddings may capture
subtler tissue distinctions than a vision-only model.

`utils.plot_nearest_neighbor_tiles()` options:
- `query_indices` — pick specific tile indices instead of auto-sampling.
- `n_queries` — number of auto-sampled queries (default `4`).
- `n_neighbors` — neighbors per query (default `5`).
- `seed` — random seed for reproducible query selection (default `42`).
- `figsize_per_tile` — figure size per tile in inches (default `1.8`).

See [Tile embeddings & PCA](https://docs.bioptimus.com/guides/workflows/embeddings-pca)
for details on interpreting embedding neighborhoods.

In [ ]:
# Show nearest-neighbor tiles in embedding space.
# Each row: a query tile followed by its 5 closest neighbors.
utils.plot_nearest_neighbor_tiles(
    embeddings,
    coords_embed,
    wsi_path=first_record.wsi_path,
    n_queries=4,  # 4 query tiles from different PCA regions
    n_neighbors=5,  # 5 nearest neighbors per query
    tile_size=224,  # 224×224 px — the model's native tile size
    mpp=0.5,  # 0.5 µm/px — the model's native resolution
)

---

## 10. Spatial Gene Expression — Single Gene Comparison

Now we compare predictions **before** (image-only) and **after** (with bulk
RNA) for a single gene. This demonstrates the impact of
[late-binding](https://docs.bioptimus.com/guides/workflows/spatial-transcriptomics#3-add-bulk-rna-multimodal-prediction).

We use **EPCAM** (Epithelial Cell Adhesion Molecule) — a well-known marker
for epithelial cells and adenocarcinoma. In lung adenocarcinoma, EPCAM is
typically overexpressed in tumor regions and absent in stroma.

Gene names in the output are **Ensembl IDs** (e.g. `ENSG00000119888` for
EPCAM). `utils.read_gene_mapping()` reads the linked bulk RNA file — which
contains both `gene_name` (symbol) and `gene_id` (Ensembl) columns — and
returns a `{symbol: ensembl_id}` dict for easy lookup.

`utils.plot_gene_overlay_comparison()` options:
- `cmap` — colormap for the heatmap (`"inferno"`, `"viridis"`, `"plasma"`,
  `"magma"`, etc.).
- `alpha` — heatmap transparency over the thumbnail (`0`=invisible,
  `1`=opaque; default `0.55`).
- `thumbnail_size` — thumbnail resolution for the overlay (default
  `(1024, 1024)`).

> **Tip:** To find the Ensembl ID for any gene, search
> [Ensembl](https://www.ensembl.org/) or use `utils.read_gene_mapping()`
> with your bulk RNA file.
>
> See [Visualizing results](https://docs.bioptimus.com/guides/get-started/visualizing-results)
> for more on working with prediction outputs.

> **Scale difference:** Predictions with bulk RNA have higher absolute
> magnitudes than image-only predictions. The color scale auto-adjusts per
> plot, so the heatmaps may look spatially similar even when the underlying
> values differ. Compare the colorbar legends, not just the colors.

In [ ]:
# Load predictions WITH bulk RNA — explicitly request the bulk_rna modality combo.
bulk_path = m_output.get_stage_path("predict", ["image", "bulk_rna"])
data_bulk = utils.load_zarr_output(bulk_path)
preds_bulk = data_bulk["outputs"]
coords_bulk = data_bulk["coords"]
gene_names = data_bulk.get("gene_names", [])

# Load image-only predictions explicitly (in case the in-memory variable was lost).
img_only_path = m_output.get_stage_path("predict", ["image"])
data_img_only = utils.load_zarr_output(img_only_path)
preds_image_only = data_img_only["outputs"]
coords_image_only = data_img_only["coords"]

print(f"Image-only predictions: {preds_image_only.shape}")
print(f"With bulk RNA:          {preds_bulk.shape}")
print(f"Output gene count:      {len(gene_names):,}")

# Sanity check: bulk-RNA predictions have the same shape as the image-only ones.
assert preds_bulk.shape == preds_image_only.shape, (
    f"Shape mismatch: bulk {preds_bulk.shape} vs image-only {preds_image_only.shape}."
)

# Build a symbol → Ensembl ID mapping from the linked bulk RNA file.
symbol_to_ensembl = utils.read_gene_mapping(
    first_record.bulk_rna_path,
    symbol_column="gene_name",
    id_column="gene_id",
    strip_version=True,
)

# Change GENE_NAME to any gene symbol present in your RNA TSV.
GENE_NAME = "EPCAM"
ensembl_id = symbol_to_ensembl[GENE_NAME]
gene_idx = gene_names.index(ensembl_id)

print(f"\nGene: {GENE_NAME} ({ensembl_id}, output index {gene_idx})")

# Side-by-side thumbnail overlay comparison.
utils.plot_gene_overlay_comparison(
    preds_bulk,
    coords_bulk,
    preds_image_only,
    coords_image_only,
    gene_idx=gene_idx,
    wsi_path=first_record.wsi_path,
    gene_name=GENE_NAME,
    label_a="With bulk RNA",
    label_b="Without bulk RNA (image only)",
    slide_id=first_record.wsi_id,
)


### Which tiles drive the signal?

The heatmap shows *where* expression is high or low, but what do those tiles
actually look like? Below we extract the tiles with the **highest** and
**lowest** predicted EPCAM expression and display them side-by-side.

In lung adenocarcinoma you would expect:
- **Highest EPCAM tiles** → dense tumor epithelium
- **Lowest EPCAM tiles** → stroma, necrosis, or immune-rich regions

`utils.plot_top_gene_tiles()` options:
- `n_top`, `n_bottom` — number of tiles to display in each row (default `5`).
- `figsize_per_tile` — figure size per tile in inches (default `2.0`).

> See the [Visualizing results guide](https://docs.bioptimus.com/guides/get-started/visualizing-results)
> for more visualization techniques.

In [ ]:
# Show tiles with the highest and lowest predicted EPCAM expression.
utils.plot_top_gene_tiles(
    preds_bulk,
    coords_bulk,
    gene_idx=gene_idx,
    wsi_path=first_record.wsi_path,
    gene_name=GENE_NAME,
    n_top=6,
    n_bottom=6,
    tile_size=224,
    mpp=0.5,
)

---

## 11. Spatial Gene Expression — Multi-Gene Panel

Examining multiple genes reveals different aspects of the tumor
microenvironment. Below we plot a panel of biologically relevant markers
for lung adenocarcinoma (TCGA-LUAD):

| Gene | Ensembl ID | Biological role |
|---|---|---|
| **EPCAM** | ENSG00000119888 | Epithelial marker — highlights tumor regions |
| **CD3E** | ENSG00000198851 | T-cell infiltrate — immune microenvironment |
| **CD8A** | ENSG00000153563 | Cytotoxic T-cells — anti-tumor immunity |
| **MKI67** | ENSG00000148773 | Proliferation marker — active cell division |
| **COL1A1** | ENSG00000108821 | Collagen — stromal/fibrotic regions |
| **SFTPC** | ENSG00000168484 | Surfactant protein C — normal alveolar type II cells |

`utils.plot_gene_panel_overlay()` options:
- `cols` — number of columns in the panel grid (default `3`).
- `cmap` — colormap (`"inferno"`, `"viridis"`, `"plasma"`, `"magma"`, etc.).
- `alpha` — heatmap transparency (`0`=invisible, `1`=opaque; default `0.55`).
- `figsize_per_plot` — size per subplot in inches (default `(6, 5)`).
- `thumbnail_size` — thumbnail resolution (default `(1024, 1024)`).

> **Note:** You can replace these genes with any of the ~6,000 in the output
> gene set. See [Spatial transcriptomics](https://docs.bioptimus.com/guides/workflows/spatial-transcriptomics)
> for guidance on gene selection and interpretation.

> **Scale difference:** The colorbar range differs between bulk RNA and
> image-only predictions. The spatial patterns are preserved, but check
> the legend to compare absolute expression levels.

In [ ]:
# Define the gene panel (symbol → Ensembl ID).
# You can modify this dict to visualize any genes of interest.
GENE_PANEL = {
    "EPCAM": "ENSG00000119888",
    "CD3E": "ENSG00000198851",
    "CD8A": "ENSG00000153563",
    "MKI67": "ENSG00000148773",
    "COL1A1": "ENSG00000108821",
    "SFTPC": "ENSG00000168484",
}

# Plot spatial heatmaps overlaid on the slide thumbnail (with bulk RNA predictions).
print("Spatial gene expression — with bulk RNA context")
print("=" * 50)
utils.plot_gene_panel_overlay(
    preds_bulk,
    coords_bulk,
    gene_names,
    GENE_PANEL,
    wsi_path=first_record.wsi_path,
    cols=3,
    cmap="inferno",
)

In [ ]:
# Compare the same panel WITHOUT bulk RNA (image-only predictions).
print("Spatial gene expression — image only (no bulk RNA)")
print("=" * 50)
utils.plot_gene_panel_overlay(
    preds_image_only,
    coords_image_only,
    gene_names,
    GENE_PANEL,
    wsi_path=first_record.wsi_path,
    cols=3,
    cmap="inferno",
)

---

## 12. (Optional) Low-Level API

The `Inference` facade is convenient for cohort-level workflows, but the SDK
also exposes a **low-level API** via `Backbone` + `SlideInference` for
single-slide control.

| Layer | Class | Use case |
|---|---|---|
| **High-level** | `Inference` | Cohort pipelines, caching, workspaces, resume |
| **Low-level** | `Backbone` + `SlideInference` | Single-slide, custom masks, custom writers |

The low-level API also exposes gene-set metadata directly:
- `model.input_gene_names` — genes the model expects in the bulk RNA input
- `model.output_gene_names` — genes in the prediction output

The `Backbone` factory supports all three backends at the low level too:
- `Backend.REMOTE` — HTTP to a running server
- `Backend.AWS` — SageMaker invoke-endpoint
- `Backend.LOCAL` — in-process `.pt2` inference on local GPU

Tissue-seg is bundled with the same SageMaker endpoint and HTTP server as
the main model — use the matching backend and the same connection parameters.

See the [Spatial transcriptomics guide](https://docs.bioptimus.com/guides/workflows/spatial-transcriptomics)
for the complete low-level workflow.

In [ ]:
# --- Low-level API example ---------------------------------------------------
# Uncomment ONE of the backend sections below to create the M-Optimus client.
#
from bioptimus.models.backbones import Backbone
from bioptimus.preprocess.wsi.models.bioptimus_mask import BioptimusTissueMaskModel
from bioptimus.preprocess.wsi.provider.tiled import TiledTissueMask
from bioptimus.inference.inference import SlideInference
from bioptimus.inference.writers import OutputFormat

# 1. Create M-Optimus client.

# --- Remote (HTTP server): ---
# model = Backbone(Models.M_OPTIMUS, backend=Backend.REMOTE, base_url=API_URL)

# --- SageMaker: ---
# model = Backbone(Models.M_OPTIMUS, backend=Backend.AWS,
#                  endpoint_name=ENDPOINT_NAME, region_name=REGION_NAME)

# --- Local (in-process GPU, no server): ---
model = Backbone(
    Models.M_OPTIMUS,
    backend=Backend.LOCAL,
    checkpoint=M_OPTIMUS_CHECKPOINT,
    device=DEVICE,
    assets_root=ASSETS_ROOT,
)

# Inspect gene sets (available on all backends).
print(f"Input genes:  {len(model.input_gene_names or [])}")
print(f"Output genes: {len(model.output_gene_names or [])}")

# 2. Create tissue mask provider.
# Tissue-seg is bundled with the same SageMaker endpoint (no extra cost)
# and the same HTTP server — use the matching backend.

# --- Remote: ---
# tissue_backbone = Backbone(Models.TISSUE_SEG, backend=Backend.REMOTE, base_url=API_URL)

# --- SageMaker (same endpoint as M-Optimus): ---
# tissue_backbone = Backbone(Models.TISSUE_SEG, backend=Backend.AWS,
#                            endpoint_name=ENDPOINT_NAME, region_name=REGION_NAME)

# --- Local: ---
tissue_backbone = Backbone(
    Models.TISSUE_SEG,
    backend=Backend.LOCAL,
    checkpoint=TISSUE_SEG_CHECKPOINT,
    device=DEVICE,
)

mask_provider = TiledTissueMask(model=BioptimusTissueMaskModel(backbone=tissue_backbone))

# 3. Run single-slide prediction (image-only).
inferrer = SlideInference(
    wsi_path=str(svs_output_path),
    model=model,
    mask_provider=mask_provider,
    mask_threshold=0.5,
)
result = inferrer.predict(
    output_path="/tmp/m_optimus_lowlevel.zarr",
    output_format=OutputFormat.ZARR,
    max_concurrency=64,  # value shown is for the local backend; use 256 for remote/aws
    mode="predict",
)

# 4. Re-run with bulk RNA.
inferrer_rna = SlideInference(
    wsi_path=str(svs_output_path),
    model=model,
    mask_provider=mask_provider,
    mask_threshold=0.5,
    bulk_rna_path=str(tsv_output_path),
    bulk_rna_gene_column="gene_id",
    bulk_rna_value_column="tpm_unstranded",
    bulk_rna_strip_version=True,
)
result_rna = inferrer_rna.predict(
    output_path="/tmp/m_optimus_lowlevel_rna.zarr",
    output_format=OutputFormat.ZARR,
    max_concurrency=64,  # value shown is for the local backend; use 256 for remote/aws
    mode="predict",
)

---

## Next Steps

You now have spatial gene expression predictions and tile embeddings from
M-Optimus. From here, you can:

| Task | Approach |
|---|---|
| **Compare with spatial assays** | Validate predictions against Visium/MERFISH ground truth |
| **Virtual spatial transcriptomics** | Generate spatial expression maps on slides without assay data |
| **Biomarker discovery** | Screen predicted genes for spatially variable expression |
| **Morphology + molecular** | Combine embeddings (morphology) with predictions (molecular) for richer analysis |
| **H-Optimus-1 features** | Extract morphology-only features — see **`h_optimus_1_tutorial.ipynb`** |

### Further reading

- [H-Optimus-1 tutorial](./h_optimus_1_tutorial.ipynb) — tile embedding
  extraction and PCA analysis
- [Spatial transcriptomics guide](https://docs.bioptimus.com/guides/workflows/spatial-transcriptomics)
  — full M-Optimus workflow documentation
- [Visualizing results](https://docs.bioptimus.com/guides/get-started/visualizing-results)
  — tile-grid heatmaps, gene overlays, and output schema
- [Choosing a model](https://docs.bioptimus.com/documentation/models/choosing-a-model)
  — H-Optimus vs. M-Optimus comparison
- [Cohorts guide](https://docs.bioptimus.com/guides/workflows/cohort)
  — multi-slide experiments with shared masks and labels